## Dataset Description

In [ ]:
# Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay

In [ ]:
# Load Data
data_path = 'Dataset/8.csv'
dataset = pd.read_csv(data_path)
display(dataset.head())
print(dataset.shape)

In [ ]:
# Basic Info
print(dataset.dtypes)
print(dataset.isnull().sum())

## Skewness & Distributions

In [ ]:
# Skewness Check
numeric_cols = dataset.select_dtypes(include=['int64', 'float64']).columns
print(dataset[numeric_cols].skew())

In [ ]:
# Histograms
dataset[numeric_cols].hist(figsize=(12,12), bins=20)
plt.tight_layout()

## Class Imbalance

In [ ]:
# Class Distribution
print(dataset['num'].value_counts())

## Preprocessing

In [ ]:
# Drop Damaged Column
df = dataset.drop(columns=['ca', 'id'])
print(df.isnull().sum())

In [ ]:
# Target and Features
X = df.drop(columns=['num'])
y = (df['num'] > 0).astype(int)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print(num_cols)
print(cat_cols)
print(y.value_counts())

In [ ]:
# Impute and Encode
numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, num_cols), ('cat', categorical_transformer, cat_cols)])
X_processed = preprocessor.fit_transform(X)
print(X_processed.shape)

In [ ]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

## Models

In [ ]:
# Train Models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}
results = {}

for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1] if hasattr(clf, 'predict_proba') else None
    report = classification_report(y_test, y_pred, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)
    results[name] = {
        'accuracy': report['accuracy'],
        'precision': report['weighted avg']['precision'],
        'recall': report['weighted avg']['recall'],
        'f1': report['weighted avg']['f1-score'],
        'roc_auc': roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
        }

In [ ]:
# Results
print(pd.DataFrame(results).T)